# Pipeline Dataset Airbnb da cidade do Rio de Janeiro

## Apresentação do Dataset
Os dados brutos (Raw) estão disponíveis no GitHub para serem carregados na camada Bronze.
* **Fonte:** [Inside Airbnb](https://insideairbnb.com/get-the-data/) (Listings.csv) e Tabela Auxiliar com as [listas dos bairros por região](https://www.estadosecapitaisdobrasil.com/lista-dos-bairros-do-rio-de-janeiro/) (Zonas).

* As segmetação da paleta de cores utilizada neste MVP é de acordo com as cores oficiais do Airbnb disponíveis no site da [US BRAND COLORS](https://usbrandcolors.com/airbnb-colors/)

* **Descrição dos dados brutos (raw) da tabela **`listings.csv`** que serão importados para a camada bronze:**

`id` — Identificador do anúncio.

`name` — Título do anúncio.

`host_id` — Identificador do host (anfitrião).

`host_name` — Nome do host.

`neighbourhood_group` — Região em que o bairro está inserido.

`neighbourhood` — Nome do bairro.

`latitude` — Latitude geográfica.

`longitude` — Longitude geográfica.

`room_type` — O tipo da acomodação.

`price` — O valor da acomodação.

`minimum_nights` — Qtd mínimo de noites para reserva.

`number_of_reviews` — Quantos reviews (avaliações) a acomodação tem.

`last_review` — Data da ultima review feita.

`reviews_per_month` — Qtd de reviews por mês.

`calculated_host_listings_count` — Qtd de anúncios que o host (anfitrião) tem na cidade.

`availability_365` — Qtd de dias que a acomodação está disponível nos próximos 365 dias.

`number_of_reviews_ltm` — Qtd de reviews que o anúncio tem nos últimos 12 meses.

`license` — Número de registro.

* **Descrição da tabela `dim_localizacao.csv` que será importada do github que será usada para indicar a região (zona) geografica da cidade que cada bairro pertence:**

`id_bairros` — Identificador do bairro

`bairro` — Nome do bairro

`zona` — Região da cidade (central, oeste, sul e norte)

**Neste MVP, desmembrei os dados do dataset `listing` ,uma tabela flat, para que criar o modelo _star schema_ a partir da arquitetura medalhão (BRONZE, SILVER & GOLD)**

### 1. Configuração e Leitura dos Dados (Camada BRONZE)

**1.1 Limpeza preventiva para ter certeza que não há nada pré-salvo (caso exista execuções anteriores)**

In [0]:
%sql
DROP TABLE IF EXISTS bronze_listings;
DROP TABLE IF EXISTS bronze_zonas_bairros;

DROP TABLE IF EXISTS dim_localizacao;
DROP TABLE IF EXISTS dim_host;
DROP TABLE IF EXISTS dim_anuncio;

DROP TABLE IF EXISTS fact_listings;

**1.2 Criação dos schemas (pastas) da arquitetura medalhão:**

* Isso garante que a estrutura 'bronze', 'silver' e 'gold' seja criada no `catálogo`

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;


**1.3 Importação da biblioteca Pandas e PySpark**

In [0]:

import pandas as pd

from pyspark.sql.functions import col, monotonically_increasing_id, regexp_replace, current_timestamp, to_date, expr, row_number, when, count, trim

**1.4 Carregando os dados na camada BRONZE**

In [0]:
# csv
url_listings_github = "https://raw.githubusercontent.com/juliafarah/MVP_Data_Engineering/refs/heads/main/listings.csv"
url_zonas_github = "https://raw.githubusercontent.com/juliafarah/MVP_Data_Engineering/refs/heads/main/dim_localizacao.csv"

# Função auxiliar para ler do GitHub e converter para Spark DataFrame
def ler_tabela_do_github(url, sep, encoding):
    
    pdf = pd.read_csv(url, sep=sep, encoding=encoding)
    pdf.columns = pdf.columns.str.strip() # remove espaços em branco antes e depois do nome da coluna
    pdf = pdf.astype(str) # coloca tudo como string pra evitar erros no spark
    df_spark = spark.createDataFrame(pdf) # converte do pandas pro spark
    return df_spark

# Listings: tabela fato
df_raw = ler_tabela_do_github(url_listings_github, sep=';', encoding='utf-8') #encoding utf-8 pra ler os erros de escrita

# Bairros/Zonas - Tabela Dimensão Raw)
df_zonas_raw = ler_tabela_do_github(url_zonas_github, sep=';', encoding='latin1') # encoding='latin1' para suportar acentos

# Salva como Tabelas Delta na camada Bronze
df_raw.write.format("delta").mode("overwrite").saveAsTable("bronze.listings")
df_zonas_raw.write.format("delta").mode("overwrite").saveAsTable("bronze.zonas_bairros")

display(df_raw.limit(15))


**1.5 Descrição inicial das colunas da tabela listings**

In [0]:
df_raw.printSchema()

### **2. Arquitetura Medalhão: Modelagem e Tratamento (Camada SILVER)**
* **Descrição**: Desmembrar a tabela flat em tabelas dimensão e fato para construir o esquema estrela (_Star Schema_).


**2.1 Dimensão Localização (`dim_localizacao`) - Camada SILVER**

> * **Objetivo:** Normalização dos nomes e gerar IDs numéricos sempre iguais para os bairros.

In [0]:
from pyspark.sql.window import Window

# Carrega a tabela bruta de zonas ingerida no passo anterior
df_bronze_zonas = spark.table("bronze.zonas_bairros")

# Seleciona bairros únicos
df_location_clean = df_bronze_zonas.select(
    trim(col("Bairro")).alias("neighbourhood"),
    trim(col("Zona")).alias("neighbourhood_group")
).distinct()

# Isso evita que o ID de um bairro mude entre execuções diferentes do pipeline.
windowSpec = Window.orderBy("neighbourhood_group", "neighbourhood")

# row_number() começa em 1, subtraímos 1 para começar do 0
df_dim_location = df_location_clean.withColumn("id_location", row_number().over(windowSpec) - 1)

# Seleção final e ordenação das colunas
df_dim_location = df_dim_location.select(
    col("id_location"),
    col("neighbourhood"),
    col("neighbourhood_group")
)

# CARGA (Silver) - Persistimos a tabela Dimensão tratada no Lakehouse
df_dim_location.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_localizacao")

display(df_dim_location.orderBy("id_location"))

In [0]:
# descricao do tipo de colunas dim_localizacao
df_dim_location.printSchema()

In [0]:
# nulos da tabela location
df_dim_location.select([count(when(col(c).isNull(), c)).alias(c) for c in df_dim_location.columns]).display()

**2.2 Dimensão Host (`dim_host`) - Camada SILVER**
* **Descrição**: Criação da tabela `dim_host` com dados do anfitrião. (**`host_id, host_name, calculated_host_listings_count`**)
* **Objetivo:** Remover duplicatas de anfitriões e selecionar métricas de perfil.
* **Tratamento:** Limpeza de IDs (remoção de sufixos .0) e validação de duplicidade e ids nulos.


In [0]:

df_dim_host = spark.table("bronze.listings") \
    .select(
        regexp_replace(col("host_id"), r"\.0$", "").alias("host_id"),
        col("host_name"),
        col("calculated_host_listings_count").cast("float").cast("int").alias("host_total_listings_count") # quantos imoveis cada host tem
    ) \
    .withColumn("host_id", regexp_replace(col("host_id"), "\.0$", "")) \
    .filter((col("host_id") != "nan") & (col("host_id").isNotNull()) & (col("host_id") != "")) \
    .dropDuplicates(["host_id"]) # Garante unicidade por ID.

# Persistir na camada Silver
df_dim_host.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_host")

display(df_dim_host)


In [0]:
window_dupe_name = Window.partitionBy("host_name")
df_dupe_check = df_dim_host.withColumn("qtd_ids_por_nome", count("host_id").over(window_dupe_name)) \
    .filter(col("qtd_ids_por_nome") > 1) \
    .select("host_name", "host_id", "host_total_listings_count", "qtd_ids_por_nome") \
    .orderBy(col("qtd_ids_por_nome").desc())

print("Auditoria: Outros hosts com mesmo nome e múltiplos IDs:")
display(df_dupe_check)

**2.3 Dimensão Anúncio (`dim_anuncio`) - Camada SILVER**
* **Objetivo:** Dados sobre o imóvel/anúcio.
* **Tratamento:** Renomeação para inglês (`ad_id`, `ad_title`) e limpeza de FKs.


In [0]:

df_dim_anuncio = spark.table("bronze.listings") \
    .select(
        col("id").alias("ad_id"),
        col("name").alias("ad_title"),
        #regexp_replace(col("host_id"), r"\.0$", "").alias("host_id"), # Mantemos host_id aqui caso queira relacionar direto, mas o ideal é via Fato
        col("room_type")
    )

# Persistir na camada Silver
df_dim_anuncio.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_anuncio")

display(df_dim_anuncio)


**2.4 Tabela Fato (`fact_listings`) - Camada SILVER**
* **Descrição:** Tabela central com métricas e chaves estrangeiras (IDs).
* **Tratamento:** Limpeza de preços (nan : NULL), conversão de tipos (try_cast) e join com dim_localizacao para pegar o id_location correto.


In [0]:
from pyspark.sql.functions import col, when, regexp_replace, round, expr

# Carregar Bronze
df_bronze = spark.table("bronze.listings")

# Carregar a dimensão localizacao (que veio do GitHub auxiliar)
df_dim_loc = spark.table("silver.dim_localizacao")

# Limpeza de Preço para remover $ e , e converter para float
df_bronze_clean = (
    df_bronze
    .withColumn(
        "price_clean", 
        when(col("price") == "nan", None)
        .otherwise(regexp_replace(col("price"), "[$,]", ""))
        .cast("float")
    )
    .withColumn(
        "host_id", 
        when(col("host_id") == "nan", None)
        .otherwise(col("host_id"))
    )
    .withColumn(
        "reviews_per_month", 
        when(
            (col("reviews_per_month") == "nan") | (col("reviews_per_month") == "NaN"), 
            None
        ).otherwise(
            round(col("reviews_per_month").cast("float"), 2)
        )
    )
)

# Join para obter o id_location (pelo nome do bairro)
df_fato = df_bronze_clean.join(
    df_dim_loc, 
    df_bronze_clean.neighbourhood == df_dim_loc.neighbourhood, 
    "left"
)

# Seleção final das colunas da Fato (Chaves + Métricas)
df_fato_final = df_fato.select(
    col("id").alias("ad_id"),           # FK para Dim Anuncio
    regexp_replace(col("host_id"), r"\.0$", "").alias("host_id"), # FK para Dim Host
    col("id_location"),                 # FK para Dim Localizacao
    col("price_clean").alias("price"),  # Métrica
    expr("try_cast(cast(minimum_nights as float) as int)").alias("minimum_nights"),
    expr("try_cast(cast(number_of_reviews as float) as int)").alias("number_of_reviews"),
    col("reviews_per_month"), # já arredondado para 2 casas decimais
    expr("try_cast(cast(availability_365 as float) as int)").alias("availability_365"),
    expr("try_cast(cast(number_of_reviews_ltm as float) as int)").alias("number_of_reviews_ltm"),
    #current_timestamp().alias("data_carga")
)

# Persistir na camada Silver : salva a tabela Fato processada no Data Lakehouse.
df_fato_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fact_listings")

display(df_fato_final)

### **3. Validação e Qualidade dos dados ainda na Camada SILVER**

**3.1 Validação Visual do Join**

* **Descrição**: Verifica se os IDs de localização na Fato correspondem aos Bairros corretos na Dimensão cuja a ordem alfabética foi feita por zona. Isto é, o índice dos bairros foram determinados de acordo com a seguinte ordem **Zona Central -> Norte -> Oeste -> Sul.**

In [0]:
df_validacao_join = spark.sql("""
    SELECT 
        f.ad_id,
        f.id_location,
        l.neighbourhood AS nome_bairro_dimensao,
        l.neighbourhood_group
    FROM silver.fact_listings f
    LEFT JOIN silver.dim_localizacao l ON f.id_location = l.id_location
    WHERE l.neighbourhood IN (
        'Copacabana', 
        'Jacarepaguá', 
        'São Cristóvão')
    ORDER BY f.id_location ASC
""")

print("Amostra de verificação do Join:")
display(df_validacao_join)

**3.2 Análise de Qualidade de Dados**
* **Objetivo**: Verificação automática de requisitos de qualidade _(nulos, preços negativos, orphan records e etc)_.

In [0]:

df_check = spark.table("silver.fact_listings")

# Verificar se temos preços negativos ou zerados
print("Contagem de preços negativos:")
print(df_check.filter(col("price") <= 0).count())

# Verificar se ficou algum registro sem id_location (orphan record)
# Isso indicaria que um bairro na tabela listings não foi encontrado na tabela auxiliar de zonas
print("Registros sem id_location (Bairro não espeficiado):")
print(df_check.filter(col("id_location").isNull()).count())

display(df_check.filter(col("id_location").isNull()))

**3.3 Investigação de Qualidade (Deep Dive**)

Análise detalhada nos 375 registros órfãos para entender a causa _(root cause)_ e análise do impacto.

In [0]:

# 1. Analise Quantitativa - cruzando a bronze com a dimensão para achar quem ficou de fora.
df_missing_neighbourhoods = spark.sql("""
    SELECT 
        b.neighbourhood as mispelling_neighbourhoods,
        count(*) as occurrences
    FROM bronze.listings b
    LEFT ANTI JOIN silver.dim_localizacao d ON b.neighbourhood = d.neighbourhood -- LEFT ANTI JOIN = LEFT JOIN + WHERE id IS NULL no Spark (só que mais rapido)
    GROUP BY b.neighbourhood
    ORDER BY occurrences DESC
""")

print("Lista de bairros dados como NULLs:")
display(df_missing_neighbourhoods)

# 2. Análise de Impacto
total_rows = df_check.count()
missing_rows = df_check.filter(col("id_location").isNull()).count()
percent_loss = (missing_rows / total_rows) * 100

print(f"Impacto da Qualidade: {missing_rows} registros perdidos de {total_rows} totais.")
print(f"Perda de Dados Geográficos: {percent_loss:.2f}%")


**3.4 Correção de Dados _(Data Cleaning_) e Recarga na camada SILVER**

* **Objetivo:** Redução dos registros órfãos a partir da aplicação da normalização dos nomes dos bairros e reprocessar a Tabela Fato.
* **Como?** Usando `CASE WHEN` para corrigir grafias (ex: Curica -> Curicica) e tratar o `nan` convertendo para um **nulo** real do banco de dados.


In [0]:

# Voltar a tabela Bronze para aplicar a correção antes do Join
df_bronze_pre_clean = df_bronze.withColumn("neighbourhood_trimmed", trim(col("neighbourhood")))

# Aplicação das regras de correção na coluna já limpa
df_bronze_corrected = df_bronze_pre_clean.withColumn("neighbourhood", 
    when(col("neighbourhood_trimmed") == "Curica", "Curicica")
    .when(col("neighbourhood_trimmed") == "Humaita", "Humaitá")
    .when(col("neighbourhood_trimmed") == "Meier", "Méier")
    .when(col("neighbourhood_trimmed") == "Jardim Botanico", "Jardim Botânico")
    .when(col("neighbourhood_trimmed") == "Osvaldo Cruz", "Oswaldo Cruz")
    .when(col("neighbourhood_trimmed") == "Turiaçú", "Turiaçu")
    .when(col("neighbourhood_trimmed") == "Freguesia (Ilha)", "Freguesia")
    .when(col("neighbourhood_trimmed") == "Freguesia (Jacarepaguá)", "Freguesia de Jacarepaguá")
    .when(col("neighbourhood_trimmed") == "nan", None)
    .otherwise(col("neighbourhood_trimmed")) # caso contrário, mantém o valor original limpo (trimmed)
).withColumn("price_clean", 
    when(col("price") == "nan", None)
    .otherwise(regexp_replace(col("price"), "[$,]", ""))
    .cast("float")
)


# Join com nomes corrigidos
df_fato_v2 = df_bronze_corrected.join(
    df_dim_loc, 
    df_bronze_corrected.neighbourhood == df_dim_loc.neighbourhood, 
    "left"
)

# 3. Seleção Final (Mesma estrutura anterior)
df_fato_final_v2 = df_fato_v2.select(
    col("id").alias("ad_id"),
    regexp_replace(col("host_id"), r"\.0$", "").alias("host_id"),
    col("id_location"),
    col("price_clean").alias("price"),
    expr("try_cast(cast(minimum_nights as float) as int)").alias("minimum_nights"),
    expr("try_cast(cast(number_of_reviews as float) as int)").alias("number_of_reviews"),
    col("reviews_per_month").cast("float").alias("reviews_per_month"),
    expr("try_cast(cast(number_of_reviews_ltm as float) as int)").alias("number_of_reviews_ltm"),
    expr("try_cast(cast(availability_365 as float) as int)").alias("availability_365"),
    current_timestamp().alias("data_carga")
)

# 4. Sobrescrita da Tabela Fato (Update Silver Layer)
df_fato_final_v2.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fact_listings")

# 5. Validação Pós-Correção
count_missing_v2 = df_fato_final_v2.filter(col("id_location").isNull()).count()
print(f"STATUS FINAL: Registros órfãos reduzidos de {missing_rows} para {count_missing_v2}.")

if count_missing_v2 == 0:
    print("SUCESSO: Qualidade de dados 100% garantida.")
else:
    print("ATENÇÃO: Ainda existem resíduos. Verificar novos casos.")

**3.5 Verificação de Registros Órfãos - camada SILVER**

* Confirmar que não há mais nomes de bairros incorretos para corrigir, apenas dados resíduo que aparecerão como `null`.

In [0]:

df_residuos = df_bronze_corrected.join(
    df_dim_loc, 
    df_bronze_corrected.neighbourhood == df_dim_loc.neighbourhood, 
    "left_anti"
).groupBy("neighbourhood").count().orderBy(col("count").desc())

display(df_residuos)

**3.6 Verificando o impacto e quantidade de nulos da coluna `ad_id` e `host_id` da `fact_listings` (tabela fato) - camada SILVER**

In [0]:

print(f"Contagem de ad_id vazios:",df_fato_final.filter(col("ad_id").isNull()).count())
print(f"Contagem de host_id vazios:",df_fato_final.filter(col("host_id").isNull()).count())

display(df_fato_final.filter(col("host_id").isNull()))

**3.7 Drop das linhas que tem `host_id` como nulo - camada SILVER**

- _Essas 177 linhas não trazem informação alguma para o analista e devem ser removidas._

In [0]:
# Drop das linhas com host_id null com SQL:
df_fato_final_v3 = spark.sql("""
    SELECT *
    FROM silver.fact_listings
    WHERE host_id IS NOT NULL
""")

# double check se todos os nulls foram removidos corretamente
print(f"Contagem de host_id nulos:", df_fato_final_v3.filter(col("host_id").isNull()).count())

# Seleção final das colunas da Fato (Chaves + Métricas)
df_fato_final_v3 = df_fato.select(
    col("id").alias("ad_id"),           # FK para Dim Anuncio
    regexp_replace(col("host_id"), r"\.0$", "").alias("host_id"), # FK para Dim Host
    col("id_location"),                 # FK para Dim Localizacao
    col("price_clean").alias("price"),  # Métrica
    expr("try_cast(cast(minimum_nights as float) as int)").alias("minimum_nights"),
    expr("try_cast(cast(number_of_reviews as float) as int)").alias("number_of_reviews"),
    col("reviews_per_month"),
    expr("try_cast(cast(availability_365 as float) as int)").alias("availability_365"),
    expr("try_cast(cast(number_of_reviews_ltm as float) as int)").alias("number_of_reviews_ltm"),
    #current_timestamp().alias("data_carga")
)


# Persistir na camada Silver : salva a tabela Fato processada no Data Lakehouse.
df_fato_final_v3.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fact_listings")

display(df_fato_final_v3)

**3.8 Verificação se o número de linhas na camada SILVER (saída) é igual ao da camada BRONZE (entrada)**

* **Objetivo**: verificar se houve perda de dados ou duplicação de dados ao avançar da camada BRONZE para a SILVER.

In [0]:
%sql
SELECT 
    (SELECT COUNT(*) FROM bronze.listings) as total_bruto,
    (SELECT COUNT(*) FROM silver.fact_listings) as total_fato,
    (SELECT COUNT(DISTINCT ad_id) FROM silver.fact_listings) as total_unicos

### **4. Carga das Tabelas Prontas para a Análise de Dados (Camada GOLD)**

**4.1: Carga - Camada Gold**

* **Descrição**: Salvar a tabela agregada final pronta para analise.

In [0]:
tabelas_star_schema = ["dim_localizacao", "dim_host", "dim_anuncio", "fact_listings"]

for tabela in tabelas_star_schema:
    df_source = spark.table(f"silver.{tabela}") # le as tabelas armazenadas na camada silver
    df_source.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"gold.{tabela}") # salva na gold


display(spark.sql("SHOW TABLES IN gold"))

### **5. Verificação Final da Carga (Data Warehouse)**

* **Descrição:** _Listagem de todas as tabelas carregadas no Hive Metastore para comprovar a etapa de Carga foi completada corretamente em todas as camadas da arquitetura medalhão._

In [0]:
print("Tabelas carregadas no Data Warehouse (Hive Metastore):")

print("\n--- Tabelas na Camada BRONZE ---")
display(spark.sql("SHOW TABLES IN bronze"))

print("--- Tabelas na Camada SILVER ---")
display(spark.sql("SHOW TABLES IN silver"))

print("--- Tabelas na Camada GOLD ---")
display(spark.sql("SHOW TABLES IN gold"))

# **Análise de Dados, utilizando linguagem SQL**

* **Objetivo:** Responder as perguntas propostas com a finalidade de extrair insights importantes para o négocio de aluguel de temporada na cidade do Rio de Janeiro.

* Tabelas utilizadas: `fact_listings`, `dim_localizacao`,`dim_anuncio`,`dim_host` da **camada GOLD**

In [0]:
# usa apenas as tabelas disponiveis na camada gold sem necessidade de colocar "gold." para acessa-las
spark.sql("USE gold") 

###**1. Qual a região do Rio tem a média de preço (ticket médio) mais cara da cidade?**


In [0]:
df_q1 = spark.sql("""
    SELECT 
        l.neighbourhood_group AS Regiao,
        ROUND(AVG(f.price), 2) AS Preco_Medio,
        count(f.ad_id) as Qtd_Anuncios
    FROM fact_listings f
    JOIN dim_localizacao l ON f.id_location = l.id_location
    GROUP BY l.neighbourhood_group
    ORDER BY Preco_Medio DESC
""")

display(df_q1)

# Visualização Gráfica
import matplotlib.pyplot as plt
import seaborn as sns

# Converte Spark DF para Pandas para plotar
pdf_q1 = df_q1.toPandas()

fig, ax1 = plt.subplots(figsize=(8, 4))

cor_barra = '#FF5A5F' # Cinza para o volume (fundo/base)
cor_linha = '#00a699' # Vermelho Airbnb para o Preço (Destaque)

bars = ax1.bar(pdf_q1['Regiao'],pdf_q1['Preco_Medio'], color=cor_barra, label='Preço Médio')
ax1.set_ylabel('Ticket Médio (R$)', fontsize=7, color='black')
ax1.tick_params(axis='y', labelcolor='black')
ax1.grid(False) # Remove grid do fundo para não poluir

# Rótulos nas barras (Volume)
ax1.bar_label(bars, labels=[f"R$ {v:,.0f}" for v in pdf_q1['Preco_Medio']], padding=3, fontsize=9, color='black')

ax2 = ax1.twinx()
line = ax2.plot(pdf_q1['Regiao'],  pdf_q1['Qtd_Anuncios'], color=cor_linha, marker='o', linewidth=2, markersize=4, label='Qtd Anúncios')
ax2.set_ylabel('Qtd Anúnicios', fontsize=8)
ax2.tick_params(axis='y', labelcolor='black')
# Define um limite um pouco maior para a linha não colar no teto do gráfico
ax2.set_ylim(0, pdf_q1['Qtd_Anuncios'].max() * 1.6)

# Rótulos na linha (anuncios)
for i, txt in enumerate(pdf_q1['Qtd_Anuncios']):
    ax2.annotate(f"{txt:.0f}", (i, pdf_q1['Qtd_Anuncios'].iloc[i]), 
                 xytext=(0, 10), textcoords='offset points', 
                 ha='center', fontsize=8, color='black')

plt.title('Ticket Médio vs. Volume de Anúncios por Região', fontsize=14, pad=20)

ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)

plt.tight_layout()
plt.show()

**Insight interessante:** Embora a Zona Sul seja o cartão-postal do Rio e concentre o maior volume de turistas, a Zona Oeste apresenta, surpreendentemente, um ticket médio 27% superior à tradicional Zona Sul.

Tal fato é explicado ao observarmos a densidade de anúncios. A Zona Oeste disponibiliza 8.034 anúncios, enquanto a Zona Sul oferta 60% a mais somando 20.210 anúncios. Essa disparidade evidencia que a Zona Sul sofre uma "diluição" do seu ticket médio, dada a gigantesca quantidade de imóveis compactos e antigos na plataforma.

Portanto, a Zona Sul mantém sua hegemonia em volume e liquidez ao abraçar o turismo de massa. Em contrapartida, a Zona Oeste se consolida como o destino de maior rentabilidade por diária, impulsionada pela oferta de imóveis mais novos e de maior metragem.

Já a Zona Norte, com um ticket médio 58% menor que a Zona Oeste e 48% menor que a Zona Sul, mostra-se um mercado maduro e estável, sustentado pela cultura boêmia tradicional e pelo fluxo de visitantes atraídos pelo emblemático Estádio do Maracanã.

Por fim, destaca-se o posicionamento estratégico da Zona Central. Mesmo apresentando um ticket médio 10% inferior ao da Zona Norte, o Centro sinaliza ser um ativo subvalorizado, visto que disponibiliza 62% mais imóveis que a zona vizinha. Somado aos incentivos públicos de revitalização, como o Reviver Centro e Porto Maravilha, a região conta com uma logística privilegiada de fácil acesso ao Aeroporto Santos Dumont, VLT e Metrô, o que a torna uma aposta inteligente tanto para pequenos investidores quanto para empresas especializadas atentos na retomada do turismo cultural e corporativo.


###**2. Quais são os 10 bairros com a maior disponibilidade de anúncios?**

In [0]:
df_q2 = spark.sql("""
SELECT 
    l.neighbourhood AS Bairro,
    l.neighbourhood_group AS Regiao,
    COUNT(f.ad_id) AS Total_Anuncios
FROM fact_listings f
LEFT JOIN dim_localizacao l ON f.id_location = l.id_location
WHERE l.neighbourhood_group IS NOT NULL
GROUP BY l.neighbourhood, l.neighbourhood_group
ORDER BY COUNT(f.ad_id) DESC
LIMIT 10;
""")

display(df_q2)

# VISUALIZACAO GRAFICA
pdf_q2 = df_q2.toPandas().sort_values('Total_Anuncios', ascending=True) # Ordena para o gráfico ficar crescente visualmente

plt.figure(figsize=(10, 5))

zone_colors_map = {
    'Zona Sul': '#FF5A5F',
    'Zona Oeste': '#00A699',
    'Zona Central': '#e51d52',
    'Zona Norte': '#ced1cb'
}
colors = [zone_colors_map.get(regiao, '#bdc3c7') for regiao in pdf_q2['Regiao']]

plt.barh(pdf_q2['Bairro'], pdf_q2['Total_Anuncios'], color=colors)


plt.title('Top 10 Bairros com Maior Oferta de Imóveis', fontsize=12)
plt.xlabel('Quantidade de Anúncios')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=zone_colors_map[z], label=z) for z in zone_colors_map]
plt.legend(handles=legend_elements, loc='lower right')

# Remove as linhas da borda superior e direita
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Rótulos nas barras
for index, value in enumerate(pdf_q2['Total_Anuncios']):
    plt.text(value, index, f' {value}', va='center')

plt.show()

**_Insight interessante:_** Evidenciando a análise anterior, nota-se a liderança da Zona Sul na preferência do mercado, ocupando 5 das 10 primeiras posições no ranking de oferta. Copacabana consolida-se como o líder absoluto em número de anúncios, detendo uma parcela de 40% da oferta entre os bairros listados, o que ratifica a fama turística da região.

A Barra da Tijuca não apenas lidera a oferta na Zona Oeste, concentrando 11% dos anúncios, como também atinge um patamar de volume tecnicamente empatado com Ipanema (apenas 63 anúncios de diferença). Esse dado confirma que o bairro se tornou uma alternativa consolidada à tradição da Zona Sul, embora o volume de ambos seja pequeno se comparado à escala massiva de Copacabana.

É importante destacar também a força do Centro. A região já supera em 15% a oferta de anúncios do Recreio dos Bandeirantes e está apenas 28% abaixo da disponibilidade da Barra da Tijuca. O grande trunfo é que, mesmo cobrando um ticket (preço) médio 66% menor que a Zona Oeste, o Centro se revela um investimento inteligente já que o valor mais acessível atrai mais hóspedes, compensando o preço menor com uma alta frequência de aluguéis (alto potencial de giro).

###**3. Quais regiões da cidade tem mais disponibilidade de apto inteiro, de apenas quarto inteiro e quarto compartilhado?**

In [0]:
df_q3 = spark.sql("""
    WITH Top10_Bairros AS (
        SELECT l.neighbourhood
        FROM fact_listings f
        JOIN dim_localizacao l ON f.id_location = l.id_location
        GROUP BY l.neighbourhood
        ORDER BY COUNT(f.ad_id) DESC
        LIMIT 10
    )
    SELECT 
        l.neighbourhood AS Bairro,
        a.room_type AS Tipo_Acomodacao,
        COUNT(f.ad_id) AS Qtd_Anuncios
    FROM fact_listings f
    JOIN dim_localizacao l ON f.id_location = l.id_location
    JOIN dim_anuncio a ON f.ad_id = a.ad_id
    JOIN Top10_Bairros t ON l.neighbourhood = t.neighbourhood
    GROUP BY l.neighbourhood, a.room_type
    ORDER BY l.neighbourhood, Qtd_Anuncios DESC
""")

display(df_q3)

# Visualização Gráfica
pdf_q3 = df_q3.toPandas()

# pivotar dados
pivot_df = pdf_q3.pivot(index='Bairro', columns='Tipo_Acomodacao', values='Qtd_Anuncios').fillna(0)

# Ordena as colunas pelo valor total (soma) de cada tipo de acomodação, do maior para o menor
col_order = pivot_df.sum().sort_values(ascending=True).index
pivot_df = pivot_df[col_order]

# Reordenar os bairros: pro o maior ficar no topo visual do gráfico
pivot_df['Total_Row'] = pivot_df.sum(axis=1)
pivot_df = pivot_df.sort_values('Total_Row', ascending=True).drop(columns='Total_Row')

custom_colors_map = {
    'Entire home/apt': '#FF5A5F',
    'Private room': '#00A699',
    'Shared room': '#FC642D',
    'Hotel room': '#484848'
}
colors = [custom_colors_map.get(col, '#bdc3c7') for col in pivot_df.columns]

ax = pivot_df.plot(kind='barh', stacked=False, figsize=(10, 5), color=colors, width=0.8)

for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}', padding=3, fontsize=6)

plt.title('Perfil de Acomodação nos 10 Bairros Mais Procurados', fontsize=12)
plt.xlabel('Quantidade de Anúncios', fontsize=8)
plt.ylabel('')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)

plt.gca().xaxis.grid(True, linestyle='--', alpha=0.3)

# Remove bordas
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Aumentando a margem direita para os rótulos não serem cortados
plt.subplots_adjust(right=0.95)

plt.tight_layout()
plt.show()

**_Insight interessante:_** A análise dos dados revela uma preferência incontestável por "Imóveis Inteiros" (Entire home/apt), que totalizam 21.812 anúncios. Isso indica que 84% da oferta nos bairros mais procurados é voltada para o aluguel de temporada com foco em privacidade total.

Entretanto, Santa Teresa figura como uma exceção a essa regra. No bairro, a predominância de imóveis inteiros cai para 57%, dividindo o mercado com os Quartos Privados, que representam expressivos 39% da oferta local. Esse dado valida quantitativamente o perfil turístico da região: um público mais jovem e mochileiros estrangeiros, dispostos a compartilhar espaços em troca da vivência boêmia e cultural característica do centro.

No extremo oposto, o Leblon é o bairro com a menor penetração de Quartos Privados, representando apenas 7% dos seus anúncios. A maioria dos demais bairros segue essa tendência de exclusividade, mantendo a oferta de quartos privados próxima ao patamar de 10%.


###**4. A plataforma está dominada por empresas especializadas ou por hosts que visam apenas em ter um renda extra?**

In [0]:
df_q5 = spark.sql("""
WITH Classificacao_Hosts AS (
    SELECT 
        host_id,
        CASE 
            WHEN host_total_listings_count = 1 THEN 'Host com 1 anúncio'
            WHEN host_total_listings_count BETWEEN 2 AND 10 THEN 'Host com 2 a 10 anúncios'
            ELSE 'Empresas especializadas'
        END AS Categoria_Host
    FROM dim_host
)

SELECT 
    c.Categoria_Host,
    COUNT(DISTINCT f.host_id) AS Qtd_Hosts,
    COUNT(f.ad_id) AS Qtd_Anuncios_Controlados,
    ROUND(AVG(f.price), 2) AS Preco_Medio_Cobrado
FROM fact_listings f
JOIN Classificacao_Hosts c ON f.host_id = c.host_id
GROUP BY 1
ORDER BY Qtd_Anuncios_Controlados DESC;
""")

#display(df_q5)

# Visualização Gráfica
pdf_q5 = df_q5.toPandas()

plt.figure(figsize=(4, 4))

plt.pie(pdf_q5['Qtd_Anuncios_Controlados'], 
        labels=pdf_q5['Categoria_Host'], 
        autopct='%1.0f%%', 
        startangle=140, 
        colors=['#00A699', '#FC642D', '#FF5A5F'])

plt.axis('equal') 
plt.show()

**_Insight interessante:_** O perfil dos hosts com apenas 1 anúncio são a maioria do mercado englobando 45% dos anúncios da cidade na plataforma. Isso corrobora a premissa original do Airbnb como uma ferramenta de economia compartilhada e geração de renda extra para moradores locais.

No entanto, vemos uma tendência clara de profissionalização. Se somarmos os hosts que administram 2 a 10 imóveis, com as "Empresas Especializadas", que administram mais de 10 imóveis, esse grupo detém 55% do marketshare, formando a maioria dos hosts.

Isso indica que o mercado carioca atingiu um nível de maturidade onde o aluguel por temporada de múltiplos imóveis se tornou um modelo de negócio predominante, superando a premissa de renda extra inicial. As grandes empresas de gestão detem uma parcela relevante de 18% do marketshare, e mesmo não monopolizando o setor, sugere uma tendência de expansão.

###**5. Quais bairros garantem maior taxa de ocupação?**

In [0]:
df_q6 = spark.sql("""
WITH Ocupacao_Anual AS(
    SELECT
       ROUND(AVG(f.availability_365), 0) AS dias_ocupados_ano,
       (365 - ROUND(AVG(f.availability_365), 0)) AS dias_livres_ano,
       l.neighbourhood AS Bairro,
       l.neighbourhood_group as Regiao,
       count(f.ad_id) as qtd_anuncios
    FROM fact_listings f
    JOIN dim_localizacao l ON f.id_location = l.id_location
    GROUP BY l.neighbourhood, l.neighbourhood_group
    HAVING COUNT(f.ad_id) >= 150
)

SELECT 
    l.neighbourhood_group AS Regiao,
    l.neighbourhood AS Bairro,
    o.dias_livres_ano AS Dias_Livres_Ano,
    o.dias_ocupados_ano AS Dias_Ocupados_Ano,
    ROUND((o.dias_ocupados_ano/365)*100,0) AS Tx_Ocupacao,
    o.qtd_anuncios AS Qtd_Anuncios
FROM fact_listings f
JOIN dim_localizacao l ON f.id_location = l.id_location
JOIN Ocupacao_Anual o ON l.neighbourhood = o.bairro
GROUP BY l.neighbourhood, o.dias_livres_ano, o.dias_ocupados_ano, l.neighbourhood_group, o.qtd_anuncios
ORDER BY Tx_Ocupacao DESC
LIMIT 15;
""")

display(df_q6)

# Visualização Gráfica

# ascending=[True, False] significa: Região A-Z, Ocupação Maior->Menor
pdf_q6 = df_q6.toPandas().sort_values(['Regiao', 'Tx_Ocupacao'], ascending=[True, False])

fig, ax1 = plt.subplots(figsize=(14, 8))

zone_colors_map = {
    'Zona Sul': '#FF5A5F',      # vermelho
    'Zona Oeste': '#00A699',    # verde
    'Zona Central': '#e51d52',  # rosa escuro
    'Zona Norte': '#ced1cb'     # cinza claro
}
colors = [zone_colors_map.get(zona, '#bdc3c7') for zona in pdf_q6['Regiao']]

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
sorted_zones = sorted([z for z in pdf_q6['Regiao'].unique() if z in zone_colors_map])

legend_elements = [Patch(facecolor=zone_colors_map[z], label=z) for z in sorted_zones]
legend_elements.append(Line2D([0], [0], color='#484848', lw=2, marker='o', label='Qtd Anúncios'))
ax1.legend(handles=legend_elements, loc='upper right')

# eixo X = bairro, eixo Y = taxa
bars = ax1.bar(pdf_q6['Bairro'], pdf_q6['Tx_Ocupacao'], color=colors, alpha=0.9, label='Taxa de Ocupação')
ax1.set_ylabel('Taxa de Ocupação Média (%)', fontsize=8, color='#484848')
ax1.set_ylim(0, 100) # 0 a 100%

# rotulo no eixo X
ax1.set_xticklabels(pdf_q6['Bairro'], rotation=45, ha='right')

# rotulo nas barras
ax1.bar_label(bars, fmt='%.0f%%', padding=3, fontsize=11)

# linha: qtd de anúncios
ax2 = ax1.twinx() # Cria o eixo secundário Y
ax2.plot(pdf_q6['Bairro'], pdf_q6['Qtd_Anuncios'], color='#484848', marker='o', linewidth=1, linestyle='-')
ax2.set_ylabel('Qtd de Anuncios', fontsize=8, color='#484848')

# rotulo na linha
for i, txt in enumerate(pdf_q6['Qtd_Anuncios']):
    ax2.annotate(txt, (i, pdf_q6['Qtd_Anuncios'].iloc[i]), 
                 xytext=(0, 8), textcoords='offset points', 
                 ha='center', fontsize=9, color='#000000')

plt.title('Taxa de Ocupação Anual por Bairro', fontsize=14, pad=20)

#legenda regioes


ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)

plt.tight_layout()
plt.show()


**_Insight interessante:_**  A região central valida a aposta na revitalização (Reviver Centro) e turismo cultural. Santa Teresa (58%) e Centro (52%) mantêm ocupação competitiva, sendo que o Centro sustenta essa taxa mesmo com alta oferta (1.974 anúncios).

Na Zona Oeste, a busca por natureza e exclusividade garante as maiores taxas da cidade. A escassez de oferta em Guaratiba (73%) e Vargem Pequena (69%) gera demanda represada, enquanto o Recreio (65%) consolida-se como a alternativa robusta à Zona Sul.

A Barra da Tijuca tem o maior inventário do gráfico (2.739 anúncios), a taxa de 56% não indica baixa procura, mas alta liquidez diluída. O volume massivo confirma o bairro como um mercado seguro e de giro constante.

Na Zona Sul, a demanda transborda para bairros "satélites". A Gávea (61%) supera a média da região ao oferecer proximidade estratégica com melhor custo-benefício. Já o Vidigal (56%) destaca-se pela busca por experiência autêntica e vista panorâmica.